# Episode 6 — XADC sensors and a data logger

The **XADC** is an analog-to-digital converter built into every Zynq chip: 12 bits,
up to 1 million samples a second. It measures the chip's own temperature and supply
voltages, and external voltages on auxiliary inputs.

On the EDGE ZYNQ, two auxiliary inputs come out on connector **J15** through a
2.32 kΩ / 1 kΩ divider, so they accept 0–3.3 V:

| J15 jumper | Sensor | Python |
| --- | --- | --- |
| pins 3–5 | LM35 temperature sensor | `xadc.lm35_celsius()` |
| pins 4–6 | LDR light sensor | `xadc.light_level()` |

In [ ]:
from pynq.overlays.base import BaseOverlay
import time

base = BaseOverlay("base.bit")
xadc = base.xadc

## 1. On-chip sensors

In [ ]:
print("Zynq die temperature: {:.1f} °C".format(xadc.temperature()))
print("VCCINT (logic core):  {:.3f} V".format(xadc.vccint()))
print("VCCAUX (auxiliary):   {:.3f} V".format(xadc.vccaux()))

## 2. The J15 sensors

The XADC returns a 16-bit number whose top 12 bits are the result. The driver converts it:
`volts = raw / 65536`, then multiplies by 3.32 to undo the divider.
The LM35 gives 10 mV per °C, so `°C = volts × 100`.

In [ ]:
print("J15 pin 3: {:.3f} V  ->  LM35 {:.1f} °C".format(xadc.j15_voltage(3), xadc.lm35_celsius()))
print("J15 pin 4: {:.3f} V  ->  light level (higher = brighter)".format(xadc.j15_voltage(4)))

## 3. A live plot

The plot redraws twice a second for 30 seconds. Warm the LM35 with your fingers or
cover the LDR with your hand.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

t, temp, light = [], [], []
t0 = time.time()
while time.time() - t0 < 30:
    t.append(time.time() - t0)
    temp.append(xadc.lm35_celsius())
    light.append(xadc.light_level())

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    ax1.plot(t, temp, color="tab:red")
    ax1.set_ylabel("LM35 (°C)")
    ax2.plot(t, light, color="tab:blue")
    ax2.set_ylabel("LDR (V)")
    ax2.set_xlabel("time (s)")
    clear_output(wait=True)
    display(fig)
    plt.close(fig)
    time.sleep(0.5)

## 4. A CSV data logger

One row a second for 60 seconds, written to `sensor_log.csv`. The current temperature
is shown on the 7-segment display. You can download the file from the file browser and
open it in Excel.

In [ ]:
import csv
from datetime import datetime

with open("sensor_log.csv", "w", newline="") as f:
    log = csv.writer(f)
    log.writerow(["time", "die_temp_C", "lm35_C", "light_V"])
    for n in range(60):
        row = [datetime.now().strftime("%H:%M:%S"),
               round(xadc.temperature(), 2),
               round(xadc.lm35_celsius(), 2),
               round(xadc.light_level(), 3)]
        log.writerow(row)
        base.seven_seg.show_number(row[2], decimals=1)
        time.sleep(1)

print(open("sensor_log.csv").read()[:300])

## 5. Plot the log

In [ ]:
rows = list(csv.DictReader(open("sensor_log.csv")))
plt.figure(figsize=(8, 3))
plt.plot([float(r["lm35_C"]) for r in rows], label="LM35")
plt.plot([float(r["die_temp_C"]) for r in rows], label="Zynq die")
plt.ylabel("°C")
plt.xlabel("seconds")
plt.legend()
plt.show()

## Try it yourself

Make a temperature alarm: beep the buzzer and light all LEDs whenever the LM35 reads
more than 2 °C above the temperature at the start. Run it for 60 seconds.

In [ ]:
# your code here